# k-Nearest Neighbors (KNN)

Nesse notebook nós iremos treinar um modelo $\text{KNN}$ em nosso dataset médico. O objetivo aqui é checar possíveis melhorias usando a abordagem de um modelo baseado em distância.

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [2]:
df = pd.read_parquet("../../data/processed/UCMF_fitted.parquet")

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   peso                    9587 non-null   Float64 
 1   altura                  8078 non-null   Int64   
 2   imc                     7708 non-null   Int64   
 3   idade                   10865 non-null  Float64 
 4   pulsos                  11657 non-null  category
 5   pa_sistolica            5131 non-null   Int64   
 6   pa_diastolica           5121 non-null   Int64   
 7   ppa                     10768 non-null  category
 8   patologia               11705 non-null  category
 9   b2                      11674 non-null  category
 10  sopro                   11682 non-null  category
 11  fc                      10986 non-null  Int64   
 12  hda1                    8565 non-null   category
 13  hda2                    11705 non-null  category
 14  sexo                    11701 non-null

## Seleção de Modelo

Antes de treinarmos o modelo em si, temos que lembrar que o $\text{KNN}$ possui um hiperparâmetro importante, que é a quantidade de vizinhos mais próximos (o valor $K$). Então precisamos escolher o melhor valor de $K$ e treinar nosso modelo com esse valor. Para isso iremos usar o `GridSearchCV`, que é um otimizador de força bruta que usar validação cruzada para escolher os melhores valores de hiperparâmetros para o nosso modelo. Além disso, vamos adicionar uma camada de redução de dimensionalidade com o $\text{PCA}$ e adicionar a quantidade de componentes principais na busca do `GridSearchCV`.

In [55]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import (
    SimpleImputer,
    KNNImputer,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
)
from sklearn.preprocessing import (
    MaxAbsScaler,
    RobustScaler
)
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.decomposition import PCA

random_state = 42

cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

def create_grid_search(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    model = KNeighborsClassifier()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", KNNImputer()),
        ("scaler", RobustScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ("scaler", MaxAbsScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough"
    )

    reducer = PCA()

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("reducer", reducer),
        ("model", model),
    ])

    param_grid = {
        "model__n_neighbors": np.arange(19, 30, 2),
        "reducer__n_components": np.arange(20, 50, 5)
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cross_validator,
    )

    return grid_search

label = LabelEncoder()

X = df.drop(["patologia"], axis="columns")
y = label.fit_transform(df["patologia"])

Agora vamos criar nosso otimizador.

In [56]:
grid_search = create_grid_search(X)

E então vamos aplicar a otimização para que ele encontre os melhores valores para os nossos hiperparâmetros.

In [57]:
grid_search.fit(X, y)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-package

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': array([19, 21..., 25, 27, 29]), 'reducer__n_components': array([20, 25..., 35, 40, 45])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for

Com a busca concluída podemos acessar os melhores parâmetros abaixo.

In [58]:
grid_search.best_params_

{'model__n_neighbors': np.int64(19), 'reducer__n_components': np.int64(25)}

Com isso podemos montar nossa pipeline, mas agora com $K = 19$ e usando $25$ componentes principais.

In [59]:
def create_pipeline(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    model = KNeighborsClassifier(
        n_neighbors=19
    )

    numeric_pipeline = Pipeline(steps=[
        ("imputer", KNNImputer()),
        ("scaler", RobustScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ("scaler", MaxAbsScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough"
    )

    reducer = PCA(
        n_components=25
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("reducer", reducer),
        ("model", model),
    ])

    return pipeline

In [60]:
pipeline = create_pipeline(X)

Agora vamos avaliar o desempenho do nosso modelo usando validação cruzada.

In [61]:
def evaluate_pipeline(X, y):
    pipeline = create_pipeline(X)

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cross_validator,
        scoring={
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1"
        },
        n_jobs=-1,
    )

    return scores

In [62]:
metrics = evaluate_pipeline(X, y)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [63]:
for metric, scores in metrics.items():
    print(f"{metric:14} = {scores.mean().round(2)}")

fit_time       = 12.17
score_time     = 4.61
test_roc_auc   = 0.9
test_accuracy  = 0.83
test_precision = 0.88
test_recall    = 0.7
test_f1        = 0.78


Como podemos ver, o $\text{KNN}$ obteve resultados piores que a regressão logística em todas as métricas e como consequência iremos rejeitar esse modelo.